<a href="https://colab.research.google.com/github/Faizaa01/Deep_Learning_Practices/blob/main/PyTorch_training_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [3]:
df.shape

(569, 33)

In [4]:
df.drop(columns=['id', 'Unnamed: 32'], inplace=True)
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [5]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)
df['diagnosis'].unique()

array(['M', 'B'], dtype=object)

In [6]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [7]:
encode = LabelEncoder()
y_train = encode.fit_transform(y_train)
y_test = encode.transform(y_test)

In [8]:
import torch

X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [9]:
type(X_train_tensor)

torch.Tensor

In [10]:
class MysimpleNN():
  def __init__(self, X):
    self.weights = torch.rand(X.shape[1],1,dtype=torch.float64, requires_grad=True)
    self.bias = torch.zeros(1, dtype=torch.float64, requires_grad=True)

  def forward(self, X):
    z = torch.matmul(X, self.weights) + self.bias
    y_pred = torch.sigmoid(z)
    return y_pred

  def loss_function(self, y_pred, y):
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred, epsilon, 1-epsilon)
    loss = -(y_train_tensor * torch.log(y_pred) + (1-y_train_tensor) * torch.log(1-y_pred)).mean()
    return loss


In [11]:
model = MysimpleNN(X_train_tensor)

In [12]:
learnong_rate = 0.1
epochs = 25

for epoch in range(epochs):
  y_pred = model.forward(X_train_tensor)

  loss = model.loss_function(y_pred, y_train_tensor)

  loss.backward()

  with torch.no_grad():
    model.weights -= learnong_rate * model.weights.grad
    model.bias -= learnong_rate * model.bias.grad

  model.weights.grad.zero_()
  model.bias.grad.zero_()
  print(f'epoch: {epoch+1}, loss: {loss.item()}')


epoch: 1, loss: 3.669004840338687
epoch: 2, loss: 3.5391113090092357
epoch: 3, loss: 3.3991503117644575
epoch: 4, loss: 3.2592336644005444
epoch: 5, loss: 3.114563784900482
epoch: 6, loss: 2.9643631008306754
epoch: 7, loss: 2.8084619505271085
epoch: 8, loss: 2.6457170481021044
epoch: 9, loss: 2.4788422382061683
epoch: 10, loss: 2.3023627904787576
epoch: 11, loss: 2.129226955890381
epoch: 12, loss: 1.9591251542992079
epoch: 13, loss: 1.7923198171814636
epoch: 14, loss: 1.6314993471020498
epoch: 15, loss: 1.4791979166549676
epoch: 16, loss: 1.3385711512229428
epoch: 17, loss: 1.2110566545626278
epoch: 18, loss: 1.101512773146923
epoch: 19, loss: 1.0110803913112285
epoch: 20, loss: 0.9398689032764782
epoch: 21, loss: 0.8865761130486263
epoch: 22, loss: 0.8484558085750117
epoch: 23, loss: 0.8219024779378923
epoch: 24, loss: 0.8033748103864954
epoch: 25, loss: 0.7900504027432759


In [13]:
with torch.no_grad():
  y_pred = model.forward(X_test_tensor)
  y_pred = (y_pred > 0.9).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'accuracy: {accuracy.item()}')

accuracy: 0.6206524968147278


In [14]:
X_train_tensor = torch.tensor(X_train_tensor, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_tensor, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_tensor, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_tensor, dtype=torch.float32)

/tmp/ipykernel_176/684536581.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train_tensor = torch.tensor(X_train_tensor, dtype=torch.float32)
/tmp/ipykernel_176/684536581.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_test_tensor = torch.tensor(X_test_tensor, dtype=torch.float32)
/tmp/ipykernel_176/684536581.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train_tensor = torch.tensor(y_train_tensor, dtype=torch.float32)
/tmp/ipykernel_176/684536581.py:4: UserWarning: To copy construct from a tensor, it is recommended to 

In [15]:
import torch.nn as nn

class MySimpleNN(nn.Module):
  def __init__(self, num_features):
    super().__init__()

    self.linear = nn.Linear(num_features, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, features):
    out = self.linear(features)
    out = self.sigmoid(out)
    return out


loss_function = nn.BCELoss()
learnong_rate = 0.1
epochs = 25
type(loss_function)

torch.nn.modules.loss.BCELoss

In [17]:
model.parameters()

<generator object Module.parameters at 0x7ca5a2bf4e40>

In [20]:
model = MySimpleNN(X_train_tensor.shape[1])

optimizer = torch.optim.SGD(model.parameters(), lr=learnong_rate)

for epoch in range(epochs):
  y_pred = model(X_train_tensor)

  loss = loss_function(y_pred, y_train_tensor.reshape(-1,1))

  optimizer.zero_grad()

  loss.backward()

  optimizer.step()


  print(f'epoch: {epoch+1}, loss: {loss.item()}')

epoch: 1, loss: 0.874382734298706
epoch: 2, loss: 0.6234409809112549
epoch: 3, loss: 0.49504828453063965
epoch: 4, loss: 0.42211654782295227
epoch: 5, loss: 0.3750053346157074
epoch: 6, loss: 0.3416609764099121
epoch: 7, loss: 0.3165368437767029
epoch: 8, loss: 0.29674383997917175
epoch: 9, loss: 0.2806260287761688
epoch: 10, loss: 0.26716315746307373
epoch: 11, loss: 0.2556898891925812
epoch: 12, loss: 0.2457527071237564
epoch: 13, loss: 0.23703090846538544
epoch: 14, loss: 0.22929051518440247
epoch: 15, loss: 0.22235669195652008
epoch: 16, loss: 0.21609561145305634
epoch: 17, loss: 0.21040302515029907
epoch: 18, loss: 0.20519627630710602
epoch: 19, loss: 0.200408935546875
epoch: 20, loss: 0.19598683714866638
epoch: 21, loss: 0.1918853372335434
epoch: 22, loss: 0.1880672723054886
epoch: 23, loss: 0.18450133502483368
epoch: 24, loss: 0.18116100132465363
epoch: 25, loss: 0.17802351713180542


In [21]:
with torch.no_grad():
  y_pred = model.forward(X_test_tensor)
  y_pred = (y_pred > 0.9).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'accuracy: {accuracy.item()}')

accuracy: 0.5646352767944336


In [23]:
X_train_tensor.shape

torch.Size([455, 30])

In [22]:
!pip install torchinfo

In [25]:
from torchinfo import summary

summary(model, input_size=(455, 30))

Layer (type:depth-idx)                   Output Shape              Param #
MySimpleNN                               [455, 1]                  --
├─Linear: 1-1                            [455, 1]                  31
├─Sigmoid: 1-2                           [455, 1]                  --
Total params: 31
Trainable params: 31
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.01
Input size (MB): 0.05
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.06